In [ ]:
import numpy as np

from ...state.mps import Mps
from ...operator.mpo import Mpo
from ...geometry.system import System

""" ... need to write something for the generic construction of tensor product operators at some point ... """

class KHMq1D:
    """ Kitaev-Heisenberg model on a cylinder with width Lx and
    circumference Ly.
    """

    def __init__(self, params=None, sites=(0,0)):
        self.model_flag = "KHMq1D"              # set an explanation flag
        self.comment = "KHMq1D, cylinder, H= Jx*X_iX_j+Jy*Y_iY_j+Jz*Z_iZ_j+Kx*X_iX_j+Ky*Y_iY_j+Kz*Z_iZ_j"       # a free comment
        if params==None:
            self.params = {"Jx":[],             # heisenberg coupling X
                           "Jy":[],             # heisenberg coupling Y
                           "Jz":[],             # heisenberg coupling Z
                           "Kx":[],             # kitaev coupling X
                           "Ky":[],             # kitaev coupling Y
                           "Kz":[]}             # kitaev coupling Z}
        else:
            self.params = params

        self.sites = sites
        self.tot_sites = np.prod(sites)         # total number of sites
        self.Lx = sites[0]                      # width Lx
        self.Ly = sites[1]                      # circumference Ly
        self.ldim = 2                           # dimension of local hilbert space
        self.channels = 6                       # the number of channels
        self.mdim = self.channels*self.Ly+2     # mpo dimension
        self.coordination_physical = (1,)       # physical coordination number
        self.coordination_virtual = (2,)        # virtual coordination number
        self.lattice = "honeycomb"              # lattice structure
        self.geometry = "cylinder"              # system geometry
        self.snake = "ring"                     # snake

        self.mps = Mps(sites=self.tot_sites,
                    local_dim=self.ldim)        # initialize an instance of an mps class

        self.mpo = Mpo(model_flag=self.model_flag,
                    sites=self.tot_sites,
                    local_dim=self.ldim,
                    mpo_dim=self.mdim,
                    channels=self.channels
                    params=self.params)         # initialize an instance of an mpo class

        self.system = System(geometry=self.geometry,
                        snake=self.snake,
                        lattice=self.lattice,
                        coordination_physical=self.coordination_physical,
                        coordination_virtual=self.coordination_virtual,
                        sites=self.sites)       # initialize an instance of an system class

    def init_pauli(self):
        """ init local site operators: spin 1/2 -> pauli matrices. """
        I = np.eye(2)
        X = np.array([[0.,1.],[1.,0.]])
        Y = np.array([[0.,-1j],[1j,0.]])
        Z = np.array([[1.,0.],[0.,-1.]])
        return(I, X, Y, Z)

    def model_dict(self):
        """ Return the model parameters. """
        return({"parameters":
            {"comment": self.comment,
            "params":self.params,
            "model_flag":self.mpo.model_flag,
            "mpo_dim":self.mpo.mpo_dim,
            "local_dim":self.mpo.local_dim,
            "coordination_physical":self.system.coordination_physical,
            "coordination_virtual":self.system.coordination_virtual,
            "lattice":self.system.lattice
        }})

    def init_rules(self, Jx, Jy, Jz, Kx, Ky, Kz):

        # init local site operators: spin 1/2 -> pauli matrices
        I, X, Y, Z = self.init_pauli()

        # rule = {"n": int(), "p": float(), "I":{"o1": operator, "i1": site}, "F":{"o2": operator, "i2": site}}
        # {"n": 0, "p": Kx, "I": [{"o1":X, "i1":None}], "F": [{"o2":X, "i2":None}]} # Kx*Xi Xj, channel = 0
        # {"n": 1, "p": Ky, "I": [{"o1":Y, "i1":None}], "F": [{"o2":Y, "i2":None}]} # Ky*Yi Yj, channel = 1
        # {"n": 2, "p": Kz, "I": [{"o1":Z, "i1":None}], "F": [{"o2":Z, "i2":None}]} # Kz*Zi Zj, channel = 2
        # {"n": 3, "p": Jx, "I": [{"o1":X, "i1":None}], "F": [{"o2":X, "i2":None}]} # Jx*Xi Xj, channel = 3
        # {"n": 4, "p": Jy, "I": [{"o1":Y, "i1":None}], "F": [{"o2":Y, "i2":None}]} # Jy*Yi Yj, channel = 4
        # {"n": 5, "p": Jz, "I": [{"o1":Z, "i1":None}], "F": [{"o2":Z, "i2":None}]} # Jz*Zi Zj, channel = 5

        rules = []
        for x in range(self.Lx):
            for y in range(self.Ly):

                ''' Heisenberg interaction '''
                # vertical heisenberg part Jv
                if y<self.Ly-1:
                    rules.append({"n": 3, "p": Jx, "I": [{"o1": X, "i1": (x,y)}], "F": [{"o2": X, "i2": (x,y+1)}]})
                    rules.append({"n": 4, "p": Jy, "I": [{"o1": Y, "i1": (x,y)}], "F": [{"o2": Y, "i2": (x,y+1)}]})
                    rules.append({"n": 5, "p": Jz, "I": [{"o1": Z, "i1": (x,y)}], "F": [{"o2": Z, "i2": (x,y+1)}]})

                # along circumference Jv
                if y==self.Ly-1:
                    rules.append({"n": 3, "p": Jx, "I": [{"o1": X, "i1": (x,0)}], "F": [{"o2": X, "i2": (x,y)}]})
                    rules.append({"n": 4, "p": Jy, "I": [{"o1": Y, "i1": (x,0)}], "F": [{"o2": Y, "i2": (x,y)}]})
                    rules.append({"n": 5, "p": Jz, "I": [{"o1": Z, "i1": (x,0)}], "F": [{"o2": Z, "i2": (x,y)}]})

                # horizontal heisenberg part Jh
                if x==0 and y%2==1:
                    rules.append({"n": 3, "p": Jx, "I": [{"o1": X, "i1": (x-1,y)}], "F":[{"o2": X, "i2": (x,y)}]})
                    rules.append({"n": 4, "p": Jy, "I": [{"o1": Y, "i1": (x-1,y)}], "F":[{"o2": Y, "i2": (x,y)}]})
                    rules.append({"n": 5, "p": Jz, "I": [{"o1": Z, "i1": (x-1,y)}], "F":[{"o2": Z, "i2": (x,y)}]})

                if x%2==0 and y%2==0:
                    rules.append({"n": 3, "p": Jx, "I": [{"o1": X, "i1": (x,y)}], "F":[{"o2": X, "i2": (x+1,y)}]})
                    rules.append({"n": 4, "p": Jy, "I": [{"o1": Y, "i1": (x,y)}], "F":[{"o2": Y, "i2": (x+1,y)}]})
                    rules.append({"n": 5, "p": Jz, "I": [{"o1": Z, "i1": (x,y)}], "F":[{"o2": Z, "i2": (x+1,y)}]})

                if x%2==1 and y%2==1:
                    rules.append({"n": 3, "p": Jx, "I": [{"o1": X, "i1": (x,y)}], "F": [{"o2": X, "i2": (x+1,y)}]})
                    rules.append({"n": 4, "p": Jy, "I": [{"o1": Y, "i1": (x,y)}], "F": [{"o2": Y, "i2": (x+1,y)}]})
                    rules.append({"n": 5, "p": Jz, "I": [{"o1": Z, "i1": (x,y)}], "F": [{"o2": Z, "i2": (x+1,y)}]})

                ''' Kitaev interaction '''
                for gamma in ["X","Y","Z"]:
                    # vertical kitaev part Kv
                    if x%2==0 and y%2==0 and gamma=="X":
                        rules.append({"n": 0, "p": Kx, "I": [{"o1": X, "i1": (x,y)}], "F": [{"o2": X, "i2": (x,y+1)}]})

                    if x%2==0 and y%2==1 and gamma=="Z":
                        rules.append({"n": 2, "p": Kz, "I": [{"o1": Z, "i1": (x,y)}], "F": [{"o2": Z, "i2": (x,y+1)}]})

                    if x%2==1 and y%2==0 and gamma=="Z":
                        rules.append({"n": 2, "p": Kz, "I": [{"o1": Z, "i1": (x,y)}], "F": [{"o2": Z, "i2": (x,y+1)}]})

                    if x%2==1 and y%2==1 and gamma=="X":
                        rules.append({"n": 0, "p": Kx, "I": [{"o1": X, "i1": (x,y)}], "F": [{"o2": X, "i2": (x,y+1)}]})

                    # along circumference Kv
                    if x%2==0 and y==self.Ly-1 and gamma=="Z":
                        rules.append({"n": 2, "p": Kz, "I": [{"o1": Z, "i1": (x,0)}], "F": [{"o2": Z, "i2": (x,y)}]})

                    if x%2==1 and y==self.Ly-1 and gamma=="X":
                        rules.append({"n": 0, "p": Kx, "I": [{"o1": X, "i1": (x,0)}], "F": [{"o2": X, "i2": (x,y)}]})

                    # horizontal kitaev part Kh
                    if x==0 and y%2==1 and gamma=="Y":
                        rules.append({"n": 1, "p": Ky, "I": [{"o1": Y, "i1":(x-1,y)}], "F": [{"o2": Y, "i2": (x,y)}]})

                    if x%2==0 and y%2==0 and gamma=="Y":
                        rules.append({"n": 1, "p": Ky, "I": [{"o1": Y, "i1": (x,y)}], "F": [{"o2": Y, "i2": (x+1,y)}]})

                    if x%2==1 and y%2==1 and gamma=="Y":
                        rules.append({"n": 1, "p": Ky, "I": [{"o1": Y, "i1": (x,y)}], "F": [{"o2": Y, "i2": (x+1,y)}]})

        # merge rules with equal initial state by adding the final states to lists
        for rule in rules:
            for other_rule in rules[rules.index(rule)+1:]:
                if rule["I"][0]["i1"] == other_rule["I"][0]["i1"] and rule["n"] == other_rule["n"]:
                    rule["F"].append(other_rule["F"][0])
                    rules.remove(other_rule)
        return rules

    def init_mpo(self, alpha):
        # init local site operators: spin 1/2 -> pauli matrices
        I, X, Y, Z = self.init_pauli()

        # shortcuts
        D = self.mdim; d = self.ldim
        Lx = self.Lx; Ly = self.Ly

        # hamiltonian parameter
        Jx, Jy, Jz = np.cos(alpha*np.pi), np.cos(alpha*np.pi), np.cos(alpha*np.pi)
        Kx, Ky, Kz = 2*np.sin(alpha*np.pi), 2*np.sin(alpha*np.pi), 2*np.sin(alpha*np.pi)
        self.params["Jx"].append(Jx)
        self.params["Jy"].append(Jy)
        self.params["Jz"].append(Jz)
        self.params["Kx"].append(Kx)
        self.params["Ky"].append(Ky)
        self.params["Kz"].append(Kz)

        # table of transition rules: 2-site rule = {"channel":channel, "hamiltonian_parameter":hamiltonian_parameter, "I":{"operator1":...,"site1":...}, "F":{"operator2":...,"site2":...}}
        transition_rules = self.init_rules(Jx, Jy, Jz, Kx, Ky, Kz)

        # init an empty container list for the mpo
        self.mpo.mpo_list = [np.zeros((D, D, d, d), dtype=complex)]*int(Lx*Ly)

        # unitary transition I->I, F->F
        for i in range(int(Lx*Ly)):
            self.mpo.mpo_list[i][0][0] = I; self.mpo.mpo_list[i][D-1][D-1] = I

        # iterating through rules and fill paths to the matrix product operator list
        for rule in transition_rules:
            n = rule["n"] # current channel
            x1, y1 = rule["I"][0]["i1"] # coordinates (xi, yi)
            i = x1*Ly+y1 # transform into single index
            if i>=0:
                self.mpo.mpo_list[i][0][n+1] = rule["p"] * rule["I"][0]["o1"] # fill MPO_list[site i] for transition I->node[channel n]
            for k in range(len(rule["F"])): # iterate through final states
                x2, y2 = rule["F"][k]["i2"] # coordinates (xj, yj)
                j = x2*Ly+y2 # transform into single index
                if j<int(Lx*Ly):
                    cd = 1 # covered distance
                    for l in np.arange(i+1, j-1):
                        self.mpo.mpo_list[l][(cd-1)*self.mpo.channels+n+1][cd*self.mpo.channels+n+1] = I
                        cd += 1
                    self.mpo.mpo_list[j][cd*self.mpo.channels+n][D-1] = rule["F"][k]["o2"]


In [ ]:
""" 1. Phase Diagram: Phase Diagram Data, q1D Kitaev-Heisenberg Model on a cylinder """

import numpy as np
import datetime as dt
import time

import sys
sys.path.append("/home/mmarahrens/tensor-network-bundle")

from tnb.algorithm.idmrg import iDMRG
from tnb.model.kitaevheisenberg.khm import KHMq1D
import matplotlib.pyplot as plt

if __name__=="__main__":

    # Model
    Lx=3; Ly=4
    khm = KHMq1D(sites=(Lx, Ly))                            # predefined model object
    #alpha_list = list(np.linspace(0.1, 2.1, 41))            # parameter list
    alpha_list = [0.7]

    # Simulation
    sim = iDMRG(model = khm)                                # initilize idmrg simulation object
    sim.chi_max = 1600                                       # set max bond dimenson
    sim.N_pre = 50                                          # set pre run without diagonalization
    sim.N_max = 500                                          # set max simulation steps
    sim.disc_weight = 1e-12                                  # set the discarded weight
    sim.eigensolver.lanczos_max = 100                       # set the max lanczos vector
    sim.init_iterables()                                    # initilize iterable list
    
    # the single calculations for this model are more complicated, therefore we
    # are producing a single output file for each point in the phase diagram to
    # avoid producing one huge file

    time_dict = {"alpha":[], "times":[]}
    for alpha in alpha_list:
        start_time = time.clock()
        print(alpha, start_time)
        # Flag for date and time
        dt_flag = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        with open("/home/mmarahrens/data_mirco/khm/khm_phasediagram" + dt_flag + ".p", "wb") as file:
            # Data storage
            sim.data.datafile = file                        # set the data file
            sim.data.dump_data(sim.simulation_dict())       # dump simulation data to file
            sim.init_mps()                                  # initilize/reset mps
            sim.init_env()                                  # initilize/reset environments
            sim.model.init_mpo(alpha)                       # initilize/reset matrix product operator
            sim.run(True, 1)                                # run the simulation
            sim.data.dump_data(sim.collect_data())          # collect data and dump it to file as dictionary
        end_time = time.clock()
        print(alpha, end_time)
        print("time:", end_time-start_time)
        time_dict["alpha"].append(alpha)
        time_dict["times"].append(end_time-start_time)